In [0]:
%sql
-- Cube 1: Store Monthly Performance
-- Grain: Store + Month
-- Metrics: Orders, revenue, cycle time, on-time delivery

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.cube_store_monthly AS
SELECT
  store_id,
  store_name,
  order_year,
  order_month,
  COUNT(order_id) as total_orders,
  COUNT(CASE WHEN order_status = 'COMPLETED' THEN order_id END) as completed_orders,
  ROUND(SUM(total_invoice_amount), 2) as total_revenue,
  ROUND(AVG(total_invoice_amount), 2) as avg_order_value,
  ROUND(AVG(days_in_shop), 2) as avg_days_in_shop,
  COUNT(CASE WHEN delivered_on_time = 1 THEN order_id END) as on_time_deliveries,
  ROUND(COUNT(CASE WHEN delivered_on_time = 1 THEN order_id END) * 100.0 / COUNT(order_id), 2) as on_time_delivery_pct
FROM mini_project_catalog.03_gold.fact_orders
GROUP BY store_id, store_name, order_year, order_month
ORDER BY store_id, order_year, order_month;

SELECT CONCAT('✅ cube_store_monthly created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.cube_store_monthly;

In [0]:
%sql
-- Cube 2: Technician Monthly Performance
-- Grain: Technician + Store + Month
-- Metrics: Orders, cycle time accuracy, on-time completion

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.cube_technician_monthly AS
SELECT
  technician_id,
  technician_name,
  store_id,
  store_name,
  order_year,
  order_month,
  COUNT(order_id) as total_orders,
  ROUND(SUM(days_in_shop), 2) as total_days_in_shop,
  ROUND(AVG(days_in_shop), 2) as avg_days_per_order,
  COUNT(CASE WHEN completed_on_time = 1 THEN order_id END) as on_time_completions,
  ROUND(COUNT(CASE WHEN completed_on_time = 1 THEN order_id END) * 100.0 / COUNT(order_id), 2) as completion_accuracy_pct,
  ROUND(AVG(CASE WHEN completed_on_time = 0 THEN ABS(DATEDIFF(actual_completion_datetime, planned_completion_datetime)) END), 2) as avg_delay_days
FROM mini_project_catalog.03_gold.fact_orders
WHERE order_status = 'COMPLETED'
  AND technician_id IS NOT NULL
GROUP BY technician_id, technician_name, store_id, store_name, order_year, order_month
ORDER BY store_id, order_year, order_month, technician_id;

SELECT CONCAT('✅ cube_technician_monthly created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.cube_technician_monthly;

In [0]:
%sql
-- Cube 3: Service Type Monthly Analysis
-- Grain: Service Type + Store + Month
-- Metrics: Orders, revenue, cycle time by stage

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.cube_service_type_monthly AS
SELECT
  service_type,
  store_id,
  store_name,
  order_year,
  order_month,
  COUNT(order_id) as total_orders,
  ROUND(SUM(total_invoice_amount), 2) as total_revenue,
  ROUND(AVG(total_invoice_amount), 2) as avg_revenue_per_order,
  ROUND(AVG(days_in_shop), 2) as avg_days_in_shop,
  ROUND(AVG(days_vehicle_in_to_work_start), 2) as avg_days_to_start,
  ROUND(AVG(days_work_start_to_completion), 2) as avg_days_in_work,
  ROUND(AVG(days_completion_to_delivery), 2) as avg_days_to_deliver
FROM mini_project_catalog.03_gold.fact_orders
WHERE order_status = 'COMPLETED'
GROUP BY service_type, store_id, store_name, order_year, order_month
ORDER BY store_id, service_type, order_year, order_month;

SELECT CONCAT('✅ cube_service_type_monthly created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.cube_service_type_monthly;

In [0]:
%sql
-- Cube 4: Manager Monthly Performance with Budget
-- Grain: Manager + Month
-- Metrics: Orders, revenue vs budget

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.cube_manager_monthly AS
WITH manager_revenue AS (
  SELECT
    st.manager_id,
    st.manager_name,
    f.order_year,
    f.order_month,
    COUNT(f.order_id) as total_orders,
    ROUND(SUM(f.total_invoice_amount), 2) as actual_revenue,
    ROUND(AVG(f.days_in_shop), 2) as avg_days_in_shop
  FROM mini_project_catalog.03_gold.fact_orders f
  JOIN mini_project_catalog.02_silver.store st ON f.store_id = st.store_id
  WHERE f.order_status = 'COMPLETED'
  GROUP BY st.manager_id, st.manager_name, f.order_year, f.order_month
),
manager_budget AS (
  SELECT
    st.manager_id,
    YEAR(b.month) as year,
    MONTH(b.month) as month,
    SUM(b.budget_amount) as budget_amount
  FROM mini_project_catalog.02_silver.ns_budget b
  JOIN mini_project_catalog.02_silver.store st ON b.ns_store_id = st.store_id
  GROUP BY st.manager_id, YEAR(b.month), MONTH(b.month)
)
SELECT
  mr.manager_id,
  mr.manager_name,
  mr.order_year,
  mr.order_month,
  mr.total_orders,
  mr.actual_revenue,
  COALESCE(mb.budget_amount, 0) as budget_amount,
  ROUND((mr.actual_revenue / NULLIF(mb.budget_amount, 0)) * 100, 2) as budget_achievement_pct,
  ROUND(mr.actual_revenue - COALESCE(mb.budget_amount, 0), 2) as variance_amount,
  mr.avg_days_in_shop
FROM manager_revenue mr
LEFT JOIN manager_budget mb 
  ON mr.manager_id = mb.manager_id 
  AND mr.order_year = mb.year 
  AND mr.order_month = mb.month
ORDER BY mr.manager_id, mr.order_year, mr.order_month;

SELECT CONCAT('✅ cube_manager_monthly created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.cube_manager_monthly;

In [0]:
%sql
-- Verify all 4 data cubes were created successfully
SELECT 'cube_store_monthly' as cube_name, COUNT(*) as record_count 
FROM mini_project_catalog.03_gold.cube_store_monthly
UNION ALL
SELECT 'cube_technician_monthly', COUNT(*) 
FROM mini_project_catalog.03_gold.cube_technician_monthly
UNION ALL
SELECT 'cube_service_type_monthly', COUNT(*) 
FROM mini_project_catalog.03_gold.cube_service_type_monthly
UNION ALL
SELECT 'cube_manager_monthly', COUNT(*) 
FROM mini_project_catalog.03_gold.cube_manager_monthly
ORDER BY cube_name;